# Credit Card Fraud Detection MLOps - Colab Training Pipeline

This notebook trains two autoencoder architectures for unsupervised anomaly detection on the Kaggle ULB Credit Card Fraud dataset.

**Models:**
1. Standard Deep Autoencoder
2. Tabular Transformer Autoencoder

## Step 1: Install Dependencies

In [ ]:
!pip install tensorflow keras pandas numpy scikit-learn joblib kaggle -q

## Step 2: Setup Kaggle API & Download Dataset

In [ ]:
from google.colab import files
import os

# Upload your kaggle.json file
print("Please upload your kaggle.json file when prompted...")
uploaded = files.upload()

# Setup Kaggle API
os.makedirs('/root/.kaggle', exist_ok=True)
os.system('cp kaggle.json /root/.kaggle/')
os.system('chmod 600 /root/.kaggle/kaggle.json')

# Download dataset
print("Downloading Kaggle ULB Credit Card Fraud dataset...")
os.system('kaggle datasets download -d mlg-ulb/creditcardfraud')
os.system('unzip -q creditcardfraud.zip')
print("Dataset downloaded successfully!")

## Step 3: Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Load dataset
df = pd.read_csv('creditcard.csv')
print(f"\nDataset shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())
print(f"\nClass distribution:")
print(df['Class'].value_counts())

## Step 4: Build Preprocessing Pipeline

In [ ]:
# Extract legitimate transactions only (unsupervised learning)
legitimate_transactions = df[df['Class'] == 0].copy()
print(f"Legitimate transactions shape: {legitimate_transactions.shape}")

# Drop the 'Class' column and keep only features
X_train = legitimate_transactions.drop('Class', axis=1)
print(f"Features shape: {X_train.shape}")
print(f"Features: {X_train.columns.tolist()}")

# Create preprocessing pipeline
# Apply StandardScaler to 'Time' and 'Amount', leave other features as is
preprocessor = ColumnTransformer(
    transformers=[
        ('scaler', StandardScaler(), ['Time', 'Amount']),
        ('passthrough', 'passthrough', [col for col in X_train.columns if col not in ['Time', 'Amount']])
    ],
    remainder='passthrough'
)

# Fit and transform the data
X_train_preprocessed = preprocessor.fit_transform(X_train)
print(f"\nPreprocessed data shape: {X_train_preprocessed.shape}")
print(f"Preprocessed data type: {type(X_train_preprocessed)}")

# Convert to dense array if sparse
if hasattr(X_train_preprocessed, 'toarray'):
    X_train_preprocessed = X_train_preprocessed.toarray()

X_train_preprocessed = np.array(X_train_preprocessed, dtype=np.float32)
print(f"Final preprocessed data shape: {X_train_preprocessed.shape}")
print(f"Data statistics - Mean: {X_train_preprocessed.mean():.4f}, Std: {X_train_preprocessed.std():.4f}")

## Step 5: Define Standard Deep Autoencoder Architecture

In [ ]:
def build_standard_autoencoder(input_dim=30, bottleneck_dim=8):
    """
    Build a standard fully-connected Deep Autoencoder.
    
    Architecture:
    Input (30) -> Dense(64, ReLU) -> Dense(32, ReLU) -> Dense(16, ReLU) -> 
    Bottleneck (8) -> Dense(16, ReLU) -> Dense(32, ReLU) -> Dense(64, ReLU) -> Output (30)
    """
    
    # Encoder
    inputs = layers.Input(shape=(input_dim,))
    
    encoded = layers.Dense(64, activation='relu')(inputs)
    encoded = layers.BatchNormalization()(encoded)
    
    encoded = layers.Dense(32, activation='relu')(encoded)
    encoded = layers.BatchNormalization()(encoded)
    
    encoded = layers.Dense(16, activation='relu')(encoded)
    encoded = layers.BatchNormalization()(encoded)
    
    # Bottleneck
    bottleneck = layers.Dense(bottleneck_dim, activation='relu', name='bottleneck')(encoded)
    
    # Decoder
    decoded = layers.Dense(16, activation='relu')(bottleneck)
    decoded = layers.BatchNormalization()(decoded)
    
    decoded = layers.Dense(32, activation='relu')(decoded)
    decoded = layers.BatchNormalization()(decoded)
    
    decoded = layers.Dense(64, activation='relu')(decoded)
    decoded = layers.BatchNormalization()(decoded)
    
    # Output layer (linear activation for reconstruction)
    outputs = layers.Dense(input_dim, activation='linear')(decoded)
    
    # Create model
    autoencoder = Model(inputs, outputs, name='Standard_Autoencoder')
    
    return autoencoder

# Build and display model
standard_ae = build_standard_autoencoder(input_dim=30, bottleneck_dim=8)
print("Standard Autoencoder Architecture:")
standard_ae.summary()

## Step 6: Define Tabular Transformer Autoencoder Architecture

In [ ]:
def build_transformer_autoencoder(input_dim=30, num_heads=4, bottleneck_dim=8, ff_dim=32):
    """
    Build a Tabular Transformer Autoencoder with Multi-Head Attention.
    
    Architecture:
    Input (30) -> Embedding -> Multi-Head Attention -> 
    Bottleneck (8) -> Multi-Head Attention -> Output (30)
    """
    
    # Encoder
    inputs = layers.Input(shape=(input_dim,))
    
    # Project input features to embedding dimension
    embedding_dim = 32
    x = layers.Dense(embedding_dim, activation='relu')(inputs)
    x = layers.LayerNormalization()(x)
    
    # Reshape for attention: (batch_size, seq_len, embedding_dim)
    # Treat each feature as a sequence element
    x = layers.Reshape((input_dim, 1))(x)
    
    # First transformer encoder block
    attention_output = layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=8
    )(x, x)
    x = layers.Add()([x, attention_output])
    x = layers.LayerNormalization()(x)
    
    # Feed-forward network
    ff_output = layers.Dense(ff_dim, activation='relu')(x)
    ff_output = layers.Dense(1)(ff_output)
    x = layers.Add()([x, ff_output])
    x = layers.LayerNormalization()(x)
    
    # Flatten and compress to bottleneck
    x = layers.Flatten()(x)
    bottleneck = layers.Dense(bottleneck_dim, activation='relu', name='bottleneck')(x)
    
    # Decoder
    # Expand from bottleneck
    x = layers.Dense(input_dim * 1, activation='relu')(bottleneck)
    x = layers.Reshape((input_dim, 1))(x)
    
    # Second transformer decoder block
    attention_output = layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=8
    )(x, x)
    x = layers.Add()([x, attention_output])
    x = layers.LayerNormalization()(x)
    
    # Feed-forward network
    ff_output = layers.Dense(ff_dim, activation='relu')(x)
    ff_output = layers.Dense(1)(ff_output)
    x = layers.Add()([x, ff_output])
    x = layers.LayerNormalization()(x)
    
    # Flatten and reconstruct
    x = layers.Flatten()(x)
    outputs = layers.Dense(input_dim, activation='linear')(x)
    
    # Create model
    transformer_ae = Model(inputs, outputs, name='Transformer_Autoencoder')
    
    return transformer_ae

# Build and display model
transformer_ae = build_transformer_autoencoder(input_dim=30, num_heads=4, bottleneck_dim=8, ff_dim=32)
print("Transformer Autoencoder Architecture:")
transformer_ae.summary()

## Step 7: Compile Models

In [ ]:
# Compile Standard Autoencoder
standard_ae.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)
print("Standard Autoencoder compiled successfully!")

# Compile Transformer Autoencoder
transformer_ae.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)
print("Transformer Autoencoder compiled successfully!")

## Step 8: Train Standard Autoencoder

In [ ]:
print("Training Standard Autoencoder...")
print(f"Training data shape: {X_train_preprocessed.shape}")

# Define early stopping
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

# Train model
history_standard = standard_ae.fit(
    X_train_preprocessed,
    X_train_preprocessed,
    epochs=100,
    batch_size=256,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

print(f"\nStandard Autoencoder training completed!")
print(f"Final training loss: {history_standard.history['loss'][-1]:.6f}")
print(f"Final validation loss: {history_standard.history['val_loss'][-1]:.6f}")

## Step 9: Train Transformer Autoencoder

In [ ]:
print("Training Transformer Autoencoder...")

# Define early stopping
early_stop_transformer = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

# Train model
history_transformer = transformer_ae.fit(
    X_train_preprocessed,
    X_train_preprocessed,
    epochs=100,
    batch_size=256,
    validation_split=0.2,
    callbacks=[early_stop_transformer],
    verbose=1
)

print(f"\nTransformer Autoencoder training completed!")
print(f"Final training loss: {history_transformer.history['loss'][-1]:.6f}")
print(f"Final validation loss: {history_transformer.history['val_loss'][-1]:.6f}")

## Step 10: Plot Training History

In [ ]:
# Create comparison plots
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Standard Autoencoder
axes[0].plot(history_standard.history['loss'], label='Training Loss', linewidth=2)
axes[0].plot(history_standard.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0].set_title('Standard Autoencoder - Training History', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (MSE)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Transformer Autoencoder
axes[1].plot(history_transformer.history['loss'], label='Training Loss', linewidth=2)
axes[1].plot(history_transformer.history['val_loss'], label='Validation Loss', linewidth=2)
axes[1].set_title('Transformer Autoencoder - Training History', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss (MSE)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=100, bbox_inches='tight')
plt.show()

print("Training history plot saved as 'training_history.png'")

## Step 11: Export Models and Preprocessor

In [ ]:
# Create output directory
import os
output_dir = 'models_output'
os.makedirs(output_dir, exist_ok=True)

# 1. Save preprocessor
preprocessor_path = os.path.join(output_dir, 'preprocessor.joblib')
joblib.dump(preprocessor, preprocessor_path)
print(f"✓ Preprocessor saved: {preprocessor_path}")

# 2. Save Standard Autoencoder
standard_ae_path = os.path.join(output_dir, 'standard_autoencoder.keras')
standard_ae.save(standard_ae_path)
print(f"✓ Standard Autoencoder saved: {standard_ae_path}")

# 3. Save Transformer Autoencoder
transformer_ae_path = os.path.join(output_dir, 'transformer_autoencoder.keras')
transformer_ae.save(transformer_ae_path)
print(f"✓ Transformer Autoencoder saved: {transformer_ae_path}")

# 4. Save training metadata
metadata = {
    'timestamp': datetime.now().isoformat(),
    'dataset': 'Kaggle ULB Credit Card Fraud',
    'legitimate_transactions': len(X_train_preprocessed),
    'input_features': 30,
    'standard_ae_bottleneck': 8,
    'transformer_ae_bottleneck': 8,
    'standard_ae_final_loss': float(history_standard.history['loss'][-1]),
    'standard_ae_final_val_loss': float(history_standard.history['val_loss'][-1]),
    'transformer_ae_final_loss': float(history_transformer.history['loss'][-1]),
    'transformer_ae_final_val_loss': float(history_transformer.history['val_loss'][-1])
}

import json
metadata_path = os.path.join(output_dir, 'training_metadata.json')
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=4)
print(f"✓ Training metadata saved: {metadata_path}")

print("\n" + "="*60)
print("All models and preprocessor exported successfully!")
print("="*60)

## Step 12: Download Files

In [ ]:
# Download all files
print("Preparing files for download...\n")

files_to_download = [
    f'{output_dir}/preprocessor.joblib',
    f'{output_dir}/standard_autoencoder.keras',
    f'{output_dir}/transformer_autoencoder.keras',
    f'{output_dir}/training_metadata.json',
    'training_history.png'
]

for file_path in files_to_download:
    if os.path.exists(file_path):
        print(f"Downloading: {file_path}")
        files.download(file_path)
    else:
        print(f"File not found: {file_path}")

print("\n" + "="*60)
print("Download complete! Save these files to your saved_models/ folder.")
print("="*60)